# Multimodal Visual Question Answering on VizWiz

**Course:** CSCI 5922 — Neural Networks | University of Colorado Boulder  
**Author:** Rhea Nair  

## Overview

This notebook implements multimodal Visual Question Answering (VQA) on the [VizWiz dataset](https://vizwiz.org/) — a benchmark of real images taken by blind users paired with spoken questions. The dataset is notably challenging due to image quality variation and ambiguous answers.

Four challenges are addressed:

| Challenge | Task | Approach |
|-----------|------|----------|
| 1 | Answerability classification | `MultiModalTransformer` (custom cross-attention architecture) |
| 2 | Answer generation | `MultiModalTransformer` with weighted loss + augmentation |
| 3 | Answerability classification (CLIP features) | `CLIPGenerator` on frozen CLIP embeddings |
| 4 | Answer generation (CLIP features) | `CLIPGeneratorV2` with cosine similarity input |

**Key results:**  
- CLIPGenerator VQA accuracy: **0.5346**  
- CLIPGeneratorV2 VQA accuracy: **0.5050**  
- End-to-end MultiModalTransformer VQA accuracy: **0.5352**


## 1. Setup & Data Loading

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import json
import random
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", DEVICE)

In [ ]:
# Download VizWiz images and annotations
CONTENT_FOLDER = "/content/vizwiz"
os.makedirs(CONTENT_FOLDER, exist_ok=True)

urls = {
    "Annotations.zip": "https://vizwiz.cs.colorado.edu/VizWiz_final/vqa_data/Annotations.zip",
    "train.zip":       "https://vizwiz.cs.colorado.edu/VizWiz_final/images/train.zip",
    "val.zip":         "https://vizwiz.cs.colorado.edu/VizWiz_final/images/val.zip",
    "test.zip":        "https://vizwiz.cs.colorado.edu/VizWiz_final/images/test.zip",
    # VizWiz_CLIP.zip: download from your course Canvas and place at /content/vizwiz/VizWiz_CLIP.zip
}

for filename, url in urls.items():
    print(f"Downloading {filename}...")
    !wget -q --show-progress "{url}" -O "{CONTENT_FOLDER}/{filename}"
    print(f"✅ {filename} done")

In [ ]:
!unzip -q /content/vizwiz/Annotations.zip -d /content/vizwiz/annotations/
!unzip -q /content/vizwiz/train.zip        -d /content/vizwiz/train/
!unzip -q /content/vizwiz/val.zip          -d /content/vizwiz/val/
!unzip -q /content/vizwiz/test.zip         -d /content/vizwiz/test/
!unzip -q /content/vizwiz/VizWiz_CLIP.zip  -d /content/vizwiz/clip/

print("Done. Disk usage:")
!df -h /content

In [ ]:
with open("/content/vizwiz/annotations/train.json") as f:
    train_ann = json.load(f)
with open("/content/vizwiz/annotations/val.json") as f:
    val_ann = json.load(f)
with open("/content/vizwiz/annotations/test.json") as f:
    test_ann = json.load(f)

print(f"Train: {len(train_ann)} | Val: {len(val_ann)} | Test: {len(test_ann)}")
print("Sample annotation:", train_ann[0])

## 2. Dataset

In [ ]:
class VizWizDataset(Dataset):
    """
    PyTorch Dataset for VizWiz VQA.
    Loads images and tokenizes questions using a word-level vocabulary.
    Subsetting is handled externally (pass a sliced annotation list).
    """
    def __init__(self, annotations, img_folder, transform=None, max_text_len=32):
        self.ann = annotations
        self.img_folder = img_folder
        self.max_text_len = max_text_len
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406],
                                 [0.229, 0.224, 0.225])
        ])
        # Build word vocab from all questions
        all_words = []
        for item in annotations:
            all_words += item["question"].lower().split()
        vocab = ["<pad>", "<unk>"] + sorted(set(all_words))
        self.word2idx = {w: i for i, w in enumerate(vocab)}
        self.vocab_size = len(vocab)

    def encode_question(self, question):
        tokens = question.lower().split()[:self.max_text_len]
        ids = [self.word2idx.get(t, 1) for t in tokens]
        ids += [0] * (self.max_text_len - len(ids))
        return torch.tensor(ids, dtype=torch.long)

    def __len__(self):
        return len(self.ann)

    def __getitem__(self, idx):
        item = self.ann[idx]
        img_path = os.path.join(self.img_folder, item["image"])
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        question = self.encode_question(item["question"])
        label = item.get("answerable", -1)
        if "answers" in item:
            ans_texts = [a["answer"] for a in item["answers"]]
            top_ans = Counter(ans_texts).most_common(1)[0][0]
        else:
            top_ans = ""
        return image, question, torch.tensor(label, dtype=torch.long), top_ans, item["image"]

In [ ]:
# Subsample training set to 10,000 examples (50% cap per assignment spec)
random.seed(42)
train_indices = random.sample(range(len(train_ann)), 10000)
train_ann_sub = [train_ann[i] for i in train_indices]

# Augmented transforms for training
aug_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_ds     = VizWizDataset(train_ann_sub, "/content/vizwiz/train/train")
train_ds_aug = VizWizDataset(train_ann_sub, "/content/vizwiz/train/train", transform=aug_transform)
val_ds_aug   = VizWizDataset(val_ann,       "/content/vizwiz/val/val",     transform=val_transform)
val_ds       = VizWizDataset(val_ann,       "/content/vizwiz/val/val")
test_ds      = VizWizDataset(test_ann,      "/content/vizwiz/test/test")

BATCH = 64
train_loader     = DataLoader(train_ds,     batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
train_loader_aug = DataLoader(train_ds_aug, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader       = DataLoader(val_ds,       batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
val_loader_aug   = DataLoader(val_ds_aug,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader      = DataLoader(test_ds,      batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
print(f"Vocab size: {train_ds.vocab_size}")

In [ ]:
# Build answer vocabulary (top-3000 most common answers)
all_answers = []
for item in train_ann_sub:
    for a in item["answers"]:
        all_answers.append(a["answer"])

ans_counter = Counter(all_answers)
TOP_K = 3000
ans_vocab = ["<unk>"] + [a for a, _ in ans_counter.most_common(TOP_K)]
ans2idx = {a: i for i, a in enumerate(ans_vocab)}
idx2ans = {i: a for a, i in ans2idx.items()}

print(f"Answer vocab size: {len(ans_vocab)}")
print("Top 10 answers:", [a for a, _ in ans_counter.most_common(10)])

## 3. Model Architecture — MultiModalTransformer

Custom multimodal architecture using:
- **Patch-based image encoder** (Conv2d with stride = patch_size, 7×7 = 49 patches)
- **Word-level text encoder** (embedding + positional encoding)
- **Separate self-attention** over image and text sequences
- **Cross-attention** where text tokens attend to image patch keys/values
- **Classification head** for answerability (Challenge 1)
- **Generation head** for closed-set answer prediction (Challenge 2)


In [ ]:
class ImageEncoder(nn.Module):
    """Splits image into 7x7=49 patches and linearly projects each to d_model."""
    def __init__(self, patch_size=32, d_model=256):
        super().__init__()
        self.patch_embed = nn.Conv2d(3, d_model, kernel_size=patch_size, stride=patch_size)
        self.pos_embed   = nn.Parameter(torch.randn(1, 49, d_model) * 0.02)

    def forward(self, x):
        x = self.patch_embed(x)           # (B, d_model, 7, 7)
        x = x.flatten(2).transpose(1, 2)  # (B, 49, d_model)
        return x + self.pos_embed


class TextEncoder(nn.Module):
    """Word embedding with learned positional encoding."""
    def __init__(self, vocab_size, d_model=256, max_len=32):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_embed = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)

    def forward(self, x):
        return self.embed(x) + self.pos_embed  # (B, max_len, d_model)


class MultiModalTransformer(nn.Module):
    """
    Multimodal transformer with cross-attention fusion.
    Supports both answerability classification and answer generation.
    """
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=2,
                 num_classes=2, ans_vocab_size=None):
        super().__init__()
        self.img_encoder = ImageEncoder(patch_size=32, d_model=d_model)
        self.txt_encoder = TextEncoder(vocab_size, d_model)

        img_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=512,
                                               dropout=0.1, batch_first=True)
        self.img_transformer = nn.TransformerEncoder(img_layer, num_layers=num_layers)

        txt_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward=512,
                                               dropout=0.1, batch_first=True)
        self.txt_transformer = nn.TransformerEncoder(txt_layer, num_layers=num_layers)

        # Cross-attention: text queries attend to image keys/values
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=0.1, batch_first=True)

        self.cls_head = nn.Sequential(
            nn.Linear(d_model, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, num_classes)
        )
        self.ans_vocab_size = ans_vocab_size
        if ans_vocab_size:
            self.gen_head = nn.Sequential(
                nn.Linear(d_model, 256), nn.ReLU(), nn.Linear(256, ans_vocab_size)
            )

    def encode(self, images, questions):
        img_feats = self.img_transformer(self.img_encoder(images))   # (B, 49, d)
        txt_feats = self.txt_transformer(self.txt_encoder(questions)) # (B, 32, d)
        fused, _  = self.cross_attn(txt_feats, img_feats, img_feats) # (B, 32, d)
        return fused.mean(dim=1)                                       # (B, d)

    def forward_cls(self, images, questions):
        return self.cls_head(self.encode(images, questions))

    def forward_gen(self, images, questions):
        return self.gen_head(self.encode(images, questions))


model = MultiModalTransformer(
    vocab_size=train_ds.vocab_size, d_model=256, nhead=8,
    num_layers=2, num_classes=2, ans_vocab_size=len(ans_vocab)
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

## 4. Training Utilities

In [ ]:
def train_epoch(model, loader, optimizer, task="cls"):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, questions, labels, top_ans, _ in loader:
        images, questions, labels = images.to(DEVICE), questions.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if task == "cls":
            logits = model.forward_cls(images, questions)
            loss   = F.cross_entropy(logits, labels)
            preds  = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
        else:
            ans_ids = torch.tensor([ans2idx.get(a, 0) for a in top_ans], dtype=torch.long).to(DEVICE)
            logits  = model.forward_gen(images, questions)
            loss    = F.cross_entropy(logits, ans_ids)
            preds   = logits.argmax(dim=1)
            correct += (preds == ans_ids).sum().item()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, task="cls"):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, questions, labels, top_ans, _ in loader:
            images, questions, labels = images.to(DEVICE), questions.to(DEVICE), labels.to(DEVICE)
            if task == "cls":
                logits = model.forward_cls(images, questions)
                loss   = F.cross_entropy(logits, labels)
                preds  = logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
            else:
                ans_ids = torch.tensor([ans2idx.get(a, 0) for a in top_ans], dtype=torch.long).to(DEVICE)
                logits  = model.forward_gen(images, questions)
                loss    = F.cross_entropy(logits, ans_ids)
                preds   = logits.argmax(dim=1)
                correct += (preds == ans_ids).sum().item()
            total_loss += loss.item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total


def vqa_accuracy(pred_answer, gold_answers):
    """Official VizWiz VQA accuracy: min(count/3, 1.0)"""
    gold_texts = [a["answer"] for a in gold_answers]
    return min(gold_texts.count(pred_answer) / 3, 1.0)

## 5. Challenge 1 — Answerability Classification (MultiModalTransformer)

Binary classification: is this image-question pair answerable?


In [ ]:
EPOCHS = 5
optimizer_cls = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler_cls = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_cls, T_max=EPOCHS)

best_val_acc = 0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer_cls, task="cls")
    val_loss,   val_acc   = eval_epoch(model,  val_loader,                  task="cls")
    scheduler_cls.step()
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/content/best_cls_model.pt")
        print(f"  ✅ Saved (val acc: {val_acc:.4f})")

print(f"\nBest val acc: {best_val_acc:.4f}")

In [ ]:
# Generate Challenge 1 predictions (test samples 100-199)
model.load_state_dict(torch.load("/content/best_cls_model.pt"))
model.eval()

test_ds_full = VizWizDataset(test_ann, "/content/vizwiz/test/test")
test_subset  = torch.utils.data.Subset(test_ds_full, list(range(100, 200)))
test_subset_loader = DataLoader(test_subset, batch_size=64, shuffle=False)

preds = []
with torch.no_grad():
    for images, questions, labels, top_ans, _ in test_subset_loader:
        logits = model.forward_cls(images.to(DEVICE), questions.to(DEVICE))
        preds.extend(logits.argmax(dim=1).cpu().tolist())

preds_tensor = torch.tensor(preds, dtype=torch.long)
print(f"Shape: {preds_tensor.shape} | Answerable: {preds_tensor.sum().item()} | Not: {(preds_tensor==0).sum().item()}")
torch.save(preds_tensor, "/content/Rhea_Nair_challenge1.pkl")
print("✅ Saved challenge 1 predictions")

## 6. Challenge 2 — Answer Generation (MultiModalTransformer)

Closed-set answer generation using frequency-weighted cross-entropy loss and augmented training data.


In [ ]:
# Frequency-weighted loss to reduce bias toward common answers
ans_counts = torch.zeros(len(ans_vocab))
for item in train_ann_sub:
    for a in item["answers"]:
        idx = ans2idx.get(a["answer"], 0)
        ans_counts[idx] += 1

ans_weights = 1.0 / (torch.log1p(ans_counts) + 1)
ans_weights = (ans_weights / ans_weights.sum() * len(ans_vocab)).to(DEVICE)

def train_epoch_weighted(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, questions, labels, top_ans, _ in loader:
        images, questions = images.to(DEVICE), questions.to(DEVICE)
        ans_ids = torch.tensor([ans2idx.get(a, 0) for a in top_ans], dtype=torch.long).to(DEVICE)
        optimizer.zero_grad()
        logits  = model.forward_gen(images, questions)
        loss    = F.cross_entropy(logits, ans_ids, weight=ans_weights)
        preds   = logits.argmax(dim=1)
        correct += (preds == ans_ids).sum().item()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

In [ ]:
model_gen = MultiModalTransformer(
    vocab_size=train_ds.vocab_size, d_model=256, nhead=8,
    num_layers=2, num_classes=2, ans_vocab_size=len(ans_vocab)
).to(DEVICE)

EPOCHS = 15
optimizer_gen = torch.optim.AdamW(model_gen.parameters(), lr=1e-4, weight_decay=0.05)
scheduler_gen = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_gen, T_max=EPOCHS)

best_val_acc_gen, patience, patience_counter = 0, 5, 0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch_weighted(model_gen, train_loader_aug, optimizer_gen)
    val_loss,   val_acc   = eval_epoch(model_gen, val_loader_aug, task="gen")
    scheduler_gen.step()
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    if val_acc > best_val_acc_gen:
        best_val_acc_gen = val_acc
        patience_counter = 0
        torch.save(model_gen.state_dict(), "/content/best_gen_model.pt")
        print(f"  ✅ Saved (val acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

print(f"\nBest gen val acc: {best_val_acc_gen:.4f}")

In [ ]:
# Evaluate VQA accuracy on validation set
model_gen.load_state_dict(torch.load("/content/best_gen_model.pt"))
model_gen.eval()

total_vqa_acc = 0
with torch.no_grad():
    for images, questions, labels, top_ans, img_names in val_loader_aug:
        logits   = model_gen.forward_gen(images.to(DEVICE), questions.to(DEVICE))
        pred_ids = logits.argmax(dim=1).cpu().tolist()
        for img_name, pred_id in zip(img_names, pred_ids):
            pred_ans = idx2ans[pred_id]
            ann = next(a for a in val_ann if a["image"] == img_name)
            total_vqa_acc += vqa_accuracy(pred_ans, ann["answers"])

print(f"MultiModalTransformer VQA accuracy: {total_vqa_acc / len(val_ann):.4f}")

In [ ]:
# Generate Challenge 2 predictions
test_subset_loader = DataLoader(
    torch.utils.data.Subset(VizWizDataset(test_ann, "/content/vizwiz/test/test"), list(range(100, 200))),
    batch_size=64, shuffle=False
)
results = []
with torch.no_grad():
    for images, questions, labels, top_ans, img_names in test_subset_loader:
        pred_ids = model_gen.forward_gen(images.to(DEVICE), questions.to(DEVICE)).argmax(dim=1).cpu().tolist()
        for img_name, pred_id in zip(img_names, pred_ids):
            results.append({"image": f"https://vizwiz.cs.colorado.edu/VizWiz_final/images/test/{img_name}",
                            "answer": idx2ans[pred_id]})

with open("/content/Rhea_Nair_challenge2.json", "w") as f:
    json.dump(results, f)
print(f"✅ Saved {len(results)} predictions")

## 7. Challenges 3 & 4 — CLIP Feature-Based Models

Using frozen CLIP embeddings (512-dim) provided with the dataset instead of raw images.


In [ ]:
# Load pre-extracted CLIP features
clip_train_img = torch.load("/content/vizwiz/clip/VizWiz_train_CLIP_Image.pkl")
clip_train_txt = torch.load("/content/vizwiz/clip/VizWiz_train_CLIP_Text.pkl")
clip_val_img   = torch.load("/content/vizwiz/clip/VizWiz_val_CLIP_Image.pkl")
clip_val_txt   = torch.load("/content/vizwiz/clip/VizWiz_val_CLIP_Text.pkl")
clip_test_img  = torch.load("/content/vizwiz/clip/VizWiz_test_CLIP_Image.pkl")
clip_test_txt  = torch.load("/content/vizwiz/clip/VizWiz_test_CLIP_Text.pkl")

print("Train img:", clip_train_img.shape)
print("Train txt:", clip_train_txt.shape)
print("Val img:",   clip_val_img.shape)

In [ ]:
class CLIPDataset(Dataset):
    def __init__(self, img_feats, txt_feats, annotations, subsample_indices=None):
        if subsample_indices is not None:
            self.img_feats = img_feats[subsample_indices]
            self.txt_feats = txt_feats[subsample_indices]
            self.ann = [annotations[i] for i in subsample_indices]
        else:
            self.img_feats = img_feats
            self.txt_feats = txt_feats
            self.ann = annotations

    def __len__(self):
        return len(self.ann)

    def __getitem__(self, idx):
        item = self.ann[idx]
        label = item.get("answerable", -1)
        if "answers" in item:
            ans_texts = [a["answer"] for a in item["answers"]]
            top_ans = Counter(ans_texts).most_common(1)[0][0]
        else:
            top_ans = ""
        return (self.img_feats[idx], self.txt_feats[idx],
                torch.tensor(label, dtype=torch.long), top_ans, item["image"])

clip_train_ds = CLIPDataset(clip_train_img, clip_train_txt, train_ann, subsample_indices=train_indices)
clip_val_ds   = CLIPDataset(clip_val_img,   clip_val_txt,   val_ann)
clip_test_ds  = CLIPDataset(clip_test_img,  clip_test_txt,  test_ann)

clip_train_loader = DataLoader(clip_train_ds, batch_size=64, shuffle=True,  num_workers=2)
clip_val_loader   = DataLoader(clip_val_ds,   batch_size=64, shuffle=False, num_workers=2)
clip_test_loader  = DataLoader(clip_test_ds,  batch_size=64, shuffle=False, num_workers=2)
print(f"CLIP Train: {len(clip_train_ds)} | Val: {len(clip_val_ds)} | Test: {len(clip_test_ds)}")

In [ ]:
class CLIPGenerator(nn.Module):
    """
    MLP-based answer generator operating on frozen 512-dim CLIP image+text features.
    Used for Challenges 3 (classification) and 4 (generation).
    """
    def __init__(self, d_in=512, d_model=256, ans_vocab_size=1001):
        super().__init__()
        self.img_proj = nn.Linear(d_in, d_model)
        self.txt_proj = nn.Linear(d_in, d_model)
        self.classifier = nn.Sequential(
            nn.Linear(d_model * 2, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),         nn.ReLU(),
            nn.Linear(256, ans_vocab_size)
        )

    def forward(self, img_feat, txt_feat):
        img = self.img_proj(img_feat)
        txt = self.txt_proj(txt_feat)
        return self.classifier(torch.cat([img, txt], dim=1))


class CLIPGeneratorV2(nn.Module):
    """
    Extended CLIPGenerator with cross-attention between projected features
    and cosine similarity as an additional input feature.
    """
    def __init__(self, d_in=512, d_model=256, nhead=4, ans_vocab_size=1001):
        super().__init__()
        self.img_proj = nn.Linear(d_in, d_model)
        self.txt_proj = nn.Linear(d_in, d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.generator = nn.Sequential(
            nn.Linear(d_model * 2 + 1, 512), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),             nn.ReLU(),
            nn.Linear(256, ans_vocab_size)
        )

    def forward(self, img_feat, txt_feat):
        cos_sim = F.cosine_similarity(img_feat, txt_feat, dim=1).unsqueeze(1)  # (B, 1)
        img = self.img_proj(img_feat).unsqueeze(1)
        txt = self.txt_proj(txt_feat).unsqueeze(1)
        x, _ = self.attn(torch.cat([img, txt], dim=1), torch.cat([img, txt], dim=1), torch.cat([img, txt], dim=1))
        x = self.norm(x).flatten(1)                      # (B, d_model*2)
        return self.generator(torch.cat([x, cos_sim], dim=1))

In [ ]:
def train_clip_epoch(model, loader, optimizer, task="cls"):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for img_f, txt_f, labels, top_ans, _ in loader:
        img_f, txt_f, labels = img_f.to(DEVICE), txt_f.to(DEVICE), labels.to(DEVICE)
        ans_ids = torch.tensor([ans2idx.get(a, 0) for a in top_ans], dtype=torch.long).to(DEVICE)
        optimizer.zero_grad()
        logits  = model(img_f, txt_f)
        loss    = F.cross_entropy(logits, labels if task == "cls" else ans_ids, weight=ans_weights)
        preds   = logits.argmax(dim=1)
        correct += (preds == (labels if task == "cls" else ans_ids)).sum().item()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

def eval_clip_epoch(model, loader, task="cls"):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for img_f, txt_f, labels, top_ans, _ in loader:
            img_f, txt_f, labels = img_f.to(DEVICE), txt_f.to(DEVICE), labels.to(DEVICE)
            ans_ids = torch.tensor([ans2idx.get(a, 0) for a in top_ans], dtype=torch.long).to(DEVICE)
            logits  = model(img_f, txt_f)
            loss    = F.cross_entropy(logits, labels if task == "cls" else ans_ids, weight=ans_weights)
            preds   = logits.argmax(dim=1)
            correct += (preds == (labels if task == "cls" else ans_ids)).sum().item()
            total_loss += loss.item()
            total      += labels.size(0)
    return total_loss / len(loader), correct / total

### Challenge 3 — CLIP Classification

In [ ]:
clip_cls_model = CLIPGenerator(ans_vocab_size=2).to(DEVICE)

EPOCHS = 20
optimizer_clip_cls = torch.optim.AdamW(clip_cls_model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler_clip_cls = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_clip_cls, T_max=EPOCHS)

best_clip_cls_acc, patience, patience_counter = 0, 5, 0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_clip_epoch(clip_cls_model, clip_train_loader, optimizer_clip_cls, task="cls")
    val_loss,   val_acc   = eval_clip_epoch(clip_cls_model,  clip_val_loader,                       task="cls")
    scheduler_clip_cls.step()
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    if val_acc > best_clip_cls_acc:
        best_clip_cls_acc = val_acc
        patience_counter  = 0
        torch.save(clip_cls_model.state_dict(), "/content/best_clip_cls_model.pt")
        print(f"  ✅ Saved (val acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
print(f"\nBest CLIP cls val acc: {best_clip_cls_acc:.4f}")

### Challenge 4 — CLIP Answer Generation (CLIPGenerator & CLIPGeneratorV2)

In [ ]:
clip_gen_model = CLIPGenerator(ans_vocab_size=len(ans_vocab)).to(DEVICE)

EPOCHS = 20
optimizer_clip_gen = torch.optim.AdamW(clip_gen_model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler_clip_gen = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_clip_gen, T_max=EPOCHS)

best_clip_gen_acc, patience, patience_counter = 0, 5, 0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_clip_epoch(clip_gen_model, clip_train_loader, optimizer_clip_gen, task="gen")
    val_loss,   val_acc   = eval_clip_epoch(clip_gen_model,  clip_val_loader,                       task="gen")
    scheduler_clip_gen.step()
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    if val_acc > best_clip_gen_acc:
        best_clip_gen_acc = val_acc
        patience_counter  = 0
        torch.save(clip_gen_model.state_dict(), "/content/best_clip_gen_model.pt")
        print(f"  ✅ Saved (val acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
print(f"\nBest CLIP gen val acc: {best_clip_gen_acc:.4f}")

In [ ]:
# CLIPGeneratorV2 — adds cosine similarity feature + cross-attention
clip_gen_v2 = CLIPGeneratorV2(ans_vocab_size=len(ans_vocab)).to(DEVICE)

EPOCHS = 20
optimizer_v2 = torch.optim.AdamW(clip_gen_v2.parameters(), lr=3e-4, weight_decay=0.05)
scheduler_v2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_v2, T_max=EPOCHS)

best_acc_v2, patience, patience_counter = 0, 5, 0
for epoch in range(EPOCHS):
    train_loss, train_acc = train_clip_epoch(clip_gen_v2, clip_train_loader, optimizer_v2, task="gen")
    val_loss,   val_acc   = eval_clip_epoch(clip_gen_v2,  clip_val_loader,                 task="gen")
    scheduler_v2.step()
    print(f"Epoch {epoch+1:02d} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}")
    if val_acc > best_acc_v2:
        best_acc_v2 = val_acc
        patience_counter = 0
        torch.save(clip_gen_v2.state_dict(), "/content/best_clip_gen_v2_model.pt")
        print(f"  ✅ Saved (val acc: {val_acc:.4f})")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break
print(f"\nBest V2 val acc: {best_acc_v2:.4f}")

## 8. Final Evaluation — VQA Accuracy

Official VizWiz VQA accuracy metric: for each prediction, score = min(# of annotators who gave that answer / 3, 1.0)

| Model | VQA Accuracy |
|-------|-------------|
| MultiModalTransformer (end-to-end) | 0.5352 |
| CLIPGenerator | 0.5346 |
| CLIPGeneratorV2 (cosine sim input) | 0.5050 |


In [ ]:
def compute_vqa_accuracy(model, val_loader, val_ann, use_clip=False):
    model.eval()
    total = 0
    with torch.no_grad():
        for batch in val_loader:
            if use_clip:
                img_f, txt_f, labels, top_ans, img_names = batch
                logits = model(img_f.to(DEVICE), txt_f.to(DEVICE))
            else:
                images, questions, labels, top_ans, img_names = batch
                logits = model.forward_gen(images.to(DEVICE), questions.to(DEVICE))
            pred_ids = logits.argmax(dim=1).cpu().tolist()
            for img_name, pred_id in zip(img_names, pred_ids):
                pred_ans = idx2ans[pred_id]
                ann = next(a for a in val_ann if a["image"] == img_name)
                total += vqa_accuracy(pred_ans, ann["answers"])
    return total / len(val_ann)

# Load best checkpoints and evaluate
model_gen.load_state_dict(torch.load("/content/best_gen_model.pt"))
clip_gen_model.load_state_dict(torch.load("/content/best_clip_gen_model.pt"))
clip_gen_v2.load_state_dict(torch.load("/content/best_clip_gen_v2_model.pt"))

acc_e2e  = compute_vqa_accuracy(model_gen,      val_loader_aug, val_ann, use_clip=False)
acc_clip = compute_vqa_accuracy(clip_gen_model,  clip_val_loader, val_ann, use_clip=True)
acc_v2   = compute_vqa_accuracy(clip_gen_v2,     clip_val_loader, val_ann, use_clip=True)

print(f"MultiModalTransformer (end-to-end) VQA accuracy: {acc_e2e:.4f}")
print(f"CLIPGenerator VQA accuracy:                      {acc_clip:.4f}")
print(f"CLIPGeneratorV2 VQA accuracy:                    {acc_v2:.4f}")